In [1]:
!pip install streamlit pyngrok requests beautifulsoup4 sentence-transformers faiss-cpu groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 36.4 MB/s eta 0:00:00


In [4]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

def load_model():

    df = pd.read_csv("/content/Real Time Hospital Data(imp).csv")

    # Print columns to help diagnose KeyError
    print("DataFrame columns:", df.columns.tolist())

    gender_encoder = LabelEncoder()
    df["Gender"] = gender_encoder.fit_transform(df["Gender"])

    # Check if 'Diabetes_Type' column exists before proceeding
    if "Diabetes_Type" not in df.columns:
        raise KeyError("Column 'Diabetes_Type' not found in the dataset. Please check the Excel file and ensure the column name is correct.")

    target_encoder = LabelEncoder()
    df["Diabetes_Type"] = target_encoder.fit_transform(df["Diabetes_Type"])

    X = df.drop("Diabetes_Type", axis=1)
    y = df["Diabetes_Type"]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=42, stratify=y
    )

    model = RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_split=2,
        random_state=42
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    metrics = {
        "Accuracy": round(accuracy_score(y_test, y_pred)*100,2),
        "Precision": round(precision_score(y_test, y_pred, average="weighted")*100,2),
        "Recall": round(recall_score(y_test, y_pred, average="weighted")*100,2),
        "F1 Score": round(f1_score(y_test, y_pred, average="weighted")*100,2)
    }

    return model, target_encoder, scaler, metrics, X_scaled, y

rf_model, target_encoder, scaler, metrics, X_full, y_full = load_model()
print("Accuracy of model:", metrics['Accuracy'])
print("Precision of model:", metrics['Precision'])
print("Recall of model:", metrics['Recall'])
print("F1 Score of model:", metrics['F1 Score'])

DataFrame columns: ['Age', 'Gender', 'BMI', 'Temperature', 'HeartRate', 'Glucose', 'BP_Systolic', 'BP_Diastolic', 'Diabetes_Type']
Accuracy of model: 96.35
Precision of model: 96.43
Recall of model: 96.35
F1 Score of model: 96.36


In [7]:
# ===============================
# 1. IMPORT LIBRARIES
# ===============================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ===============================
# 2. LOAD DATASETS
# ===============================
kaggle_df = pd.read_csv("/content/Kaggle dataset.csv")
realtime_df = pd.read_csv("/content/Real Time Hospital Data(imp).csv")

print("Kaggle Shape (before processing):", kaggle_df.shape)
print("Realtime Shape (before processing):", realtime_df.shape)

# ===============================
# 3. STANDARDIZE COLUMN NAMES
# ===============================
kaggle_df.columns = kaggle_df.columns.str.strip().str.lower()
realtime_df.columns = realtime_df.columns.str.strip().str.lower()

# ===============================
# 4. SEPARATE TARGET FROM KAGGLE_DF AND IDENTIFY COMMON FEATURES
# ===============================
# Define the target column name (must exist in kaggle_df)
target_column = "outcome"

if target_column not in kaggle_df.columns:
    raise KeyError(f"Target column '{target_column}' not found in Kaggle dataset after standardization. Available columns: {kaggle_df.columns.tolist()}")

# Separate target variable from Kaggle dataset
y_kaggle = kaggle_df[target_column]
kaggle_features_df = kaggle_df.drop(target_column, axis=1)

# Identify common feature columns between kaggle_features_df and realtime_df
common_feature_cols = list(set(kaggle_features_df.columns) & set(realtime_df.columns))

print("Common Feature Columns:", common_feature_cols)

# Filter both feature dataframes to keep only common feature columns
kaggle_features_df = kaggle_features_df[common_feature_cols]
realtime_features_df = realtime_df[common_feature_cols]

print("Kaggle Features Shape (after common cols):", kaggle_features_df.shape)
print("Realtime Features Shape (after common cols):", realtime_features_df.shape)

# ===============================
# 5. HANDLE MISSING VALUES
# ===============================
kaggle_features_df = kaggle_features_df.fillna(kaggle_features_df.median(numeric_only=True))
realtime_features_df = realtime_features_df.fillna(realtime_features_df.median(numeric_only=True))

# ===============================
# 6. ENCODE CATEGORICAL DATA
# ===============================
le = LabelEncoder()

# Fit and transform categorical columns for kaggle_features_df
for col in kaggle_features_df.columns:
    if kaggle_features_df[col].dtype == 'object':
        kaggle_features_df[col] = le.fit_transform(kaggle_features_df[col].astype(str))

# Transform categorical columns for realtime_features_df (using the same encoder if possible, or re-fit if new categories)
# For simplicity, using a new encoder or fitting again for realtime_df for now.
# A more robust solution might require a shared LabelEncoder or handling unseen labels.
le_realtime = LabelEncoder()
for col in realtime_features_df.columns:
    if realtime_features_df[col].dtype == 'object':
        realtime_features_df[col] = le_realtime.fit_transform(realtime_features_df[col].astype(str))


# Encode the target variable for y_kaggle
y_kaggle_encoded = le.fit_transform(y_kaggle.astype(str)) # Using the same le, assuming it's for 'diabetes-type'


# ===============================
# 7. TRAIN-TEST SPLIT (using kaggle_df for training)
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    kaggle_features_df, y_kaggle_encoded, test_size=0.2, random_state=42, stratify=y_kaggle_encoded
)

# ===============================
# 8. TRAIN RANDOM FOREST
# ===============================
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

model.fit(X_train, y_train)

# ===============================
# 9. PREDICTIONS
# ===============================
y_pred = model.predict(X_test)

# ===============================
# 10. EVALUATION
# ===============================
accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy (Kaggle Test Set):", accuracy)
print("\nClassification Report (Kaggle Test Set):\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix (Kaggle Test Set):\n", confusion_matrix(y_test, y_pred))

# ===============================
# 11. PREDICT ON REALTIME DATA
# ===============================
print("\nPredicting on Realtime Data:")
realtime_predictions_encoded = model.predict(realtime_features_df)
realtime_predictions = le.inverse_transform(realtime_predictions_encoded) # Convert back to original labels
print(f"First 5 predictions on realtime data: {realtime_predictions[:5]}")

Kaggle Shape (before processing): (768, 9)
Realtime Shape (before processing): (30000, 9)
Common Feature Columns: ['glucose', 'bmi', 'age']
Kaggle Features Shape (after common cols): (768, 3)
Realtime Features Shape (after common cols): (30000, 3)

Accuracy (Kaggle Test Set): 0.7402597402597403

Classification Report (Kaggle Test Set):
               precision    recall  f1-score   support

           0       0.78      0.84      0.81       100
           1       0.65      0.56      0.60        54

    accuracy                           0.74       154
   macro avg       0.71      0.70      0.70       154
weighted avg       0.73      0.74      0.73       154


Confusion Matrix (Kaggle Test Set):
 [[84 16]
 [24 30]]

Predicting on Realtime Data:
First 5 predictions on realtime data: ['0' '0' '0' '1' '1']


In [8]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

def load_model():

    df = pd.read_csv("/content/Real Time Hospital Data(imp).csv")

    gender_encoder = LabelEncoder()
    df["Gender"] = gender_encoder.fit_transform(df["Gender"])

    target_encoder = LabelEncoder()
    df["Diabetes_Type"] = target_encoder.fit_transform(df["Diabetes_Type"])

    X = df.drop("Diabetes_Type", axis=1)
    y = df["Diabetes_Type"]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=42, stratify=y
    )

    model = RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_split=2,
        random_state=42
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    metrics = {
        "Accuracy": round(accuracy_score(y_test, y_pred)*100,2),
        "Precision": round(precision_score(y_test, y_pred, average="weighted")*100,2),
        "Recall": round(recall_score(y_test, y_pred, average="weighted")*100,2),
        "F1 Score": round(f1_score(y_test, y_pred, average="weighted")*100,2)
    }

    return model, target_encoder, scaler, metrics, X_scaled, y

rf_model, target_encoder, scaler, metrics, X_full, y_full = load_model()
print("Accuracy of model:", metrics['Accuracy'])
print("Precision of model:", metrics['Precision'])
print("Recall of model:", metrics['Recall'])
print("F1 Score of model:", metrics['F1 Score'])


Accuracy of model: 96.35
Precision of model: 96.43
Recall of model: 96.35
F1 Score of model: 96.36


In [9]:
%%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import os
import sqlite3
import requests
from datetime import datetime
from bs4 import BeautifulSoup

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sentence_transformers import SentenceTransformer
import faiss
from groq import Groq


# ---------------------------
# DATABASE SETUP
# ---------------------------
DB_PATH = "diabetes_app.db"

def create_tables():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS patients (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        Time TEXT,
        Age INTEGER,
        Gender TEXT,
        BMI REAL,
        Glucose REAL,
        Prediction TEXT,
        Risk INTEGER,
        Suggestion TEXT,
        Diet TEXT
    )
    """)

    conn.commit()
    conn.close()

create_tables()


# ---------------------------
# SESSION MEMORY
# ---------------------------
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

if "patient_history" not in st.session_state:
    st.session_state.patient_history = []


# ---------------------------
# LOAD & TRAIN MODEL
# ---------------------------
@st.cache_resource
def load_model():

    df = pd.read_csv("/content/Real Time Hospital Data(imp).csv")

    gender_encoder = LabelEncoder()
    df["Gender"] = gender_encoder.fit_transform(df["Gender"])

    target_encoder = LabelEncoder()
    df["Diabetes_Type"] = target_encoder.fit_transform(df["Diabetes_Type"])

    X = df.drop("Diabetes_Type", axis=1)
    y = df["Diabetes_Type"]

    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    model = RandomForestClassifier(n_estimators=500, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    metrics = {
        "Accuracy": round(accuracy_score(y_test, y_pred)*100,2),
        "Precision": round(precision_score(y_test, y_pred, average="weighted")*100,2),
        "Recall": round(recall_score(y_test, y_pred, average="weighted")*100,2),
        "F1 Score": round(f1_score(y_test, y_pred, average="weighted")*100,2)
    }

    return model, target_encoder, scaler, metrics


rf_model, target_encoder, scaler, metrics = load_model()


# ---------------------------
# RAG SETUP (MULTI-SOURCE)
# ---------------------------
@st.cache_resource
def load_rag():

    dataset = [
        {"id": "WHO", "url": "https://www.who.int/news-room/fact-sheets/detail/diabetes"},
        {"id": "CDC", "url": "https://www.cdc.gov/diabetes/basics/diabetes.html"},
        {"id": "NIDDK", "url": "https://www.niddk.nih.gov/health-information/diabetes/overview/what-is-diabetes"},
        {"id": "MAYO", "url": "https://www.mayoclinic.org/diseases-conditions/diabetes/symptoms-causes/syc-20371444"}
    ]

    def fetch_text(url):
        try:
            res = requests.get(url, timeout=10)
            soup = BeautifulSoup(res.text, "html.parser")

            for tag in soup(["script", "style", "nav", "footer"]):
                tag.decompose()

            text = soup.get_text(separator=" ")
            return " ".join(text.split())

        except:
            return ""

    def chunk_text(text, size=300):
        words = text.split()
        return [" ".join(words[i:i+size]) for i in range(0, len(words), size)]

    embed_model = SentenceTransformer("all-MiniLM-L6-v2")

    documents = []
    metadata = []

    for item in dataset:
        text = fetch_text(item["url"])
        chunks = chunk_text(text)

        for chunk in chunks:
            documents.append(chunk)
            metadata.append(item["id"])

    embeddings = embed_model.encode(documents)

    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(np.array(embeddings))

    return embed_model, index, documents, metadata


embed_model, index, documents, metadata = load_rag()


# ---------------------------
# LLM SETUP
# ---------------------------
client = Groq(api_key="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")
#client = Groq(api_key=os.environ.get("gsk_7AE5yNcRoKK813pFNDPHWGdyb3FYQkg7VkaGJtsb476ChxrxZDp"))

def query_llama(messages):
    response = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=messages,
        temperature=0.6,
        max_tokens=500
    )
    return response.choices[0].message.content


# ---------------------------
# BMI
# ---------------------------
def calculate_bmi(weight, height_cm):
    return round(weight / ((height_cm/100)**2), 2)


# ---------------------------
# RISK CALCULATION
# ---------------------------
def calculate_risk(glucose, bmi, age, heart):
    risk = 0

    if glucose > 140:
        risk += 40
    elif glucose > 110:
        risk += 25

    if bmi > 30:
        risk += 25
    elif bmi > 25:
        risk += 15

    if age > 45:
        risk += 20

    if heart > 100:
        risk += 10

    return min(risk, 100)


# ---------------------------
# DIET PLAN
# ---------------------------
def diet_plan(glucose):

    if glucose > 180:
        return "Avoid sweets, sugar, and white rice."

    elif glucose > 140:
        return "Reduce carbs, eat vegetables."

    else:
        return "Balanced healthy diet."


# ---------------------------
# SUGGESTIONS
# ---------------------------
def simple_suggestion(pred, glucose, bmi, heart):

    if pred == "Type 2":
        return "Exercise daily, reduce sugar intake."

    elif pred == "Type 1":
        return "Consult doctor, insulin required."

    elif pred == "Gestational":
        return "Monitor pregnancy diet carefully."

    elif glucose > 180:
        return "⚠️ Very high glucose. Consult doctor."

    return "Healthy lifestyle recommended."


# ---------------------------
# SAVE PATIENT
# ---------------------------
def save_patient(record):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
    INSERT INTO patients (Time, Age, Gender, BMI, Glucose, Prediction, Risk, Suggestion, Diet)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        str(datetime.now()),
        record["Age"],
        record["Gender"],
        record["BMI"],
        record["Glucose"],
        record["Prediction"],
        record["Risk"],
        record["Suggestion"],
        record["Diet"]
    ))

    conn.commit()
    conn.close()


def load_patients():
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query("SELECT * FROM patients", conn)
    conn.close()
    return df


# ---------------------------
# PREDICTION
# ---------------------------
def predict_diabetes(age, gender, bmi, temp, heart, glucose, sys, dia):

    gender_encoded = 1 if gender == "Male" else 0

    input_data = [[age, gender_encoded, bmi, temp, heart, glucose, sys, dia]]
    input_data = scaler.transform(input_data)

    pred = rf_model.predict(input_data)

    return target_encoder.inverse_transform(pred)[0]


# ---------------------------
# RAG ANSWER
# ---------------------------
def rag_answer(question, patient_data):

    query_embedding = embed_model.encode([question])
    D, I = index.search(np.array(query_embedding), k=3)

    context = "\n\n".join([documents[i] for i in I[0]])
    sources = [metadata[i] for i in I[0]]

    messages = [
        {"role": "system", "content": "Answer using medical context only."},
        {"role": "system", "content": f"Patient Data: {patient_data}"},
        {"role": "system", "content": f"Context: {context}"}
    ]

    messages.extend(st.session_state.chat_history)
    messages.append({"role": "user", "content": question})

    answer = query_llama(messages)

    st.session_state.chat_history.append({"role": "user", "content": question})
    st.session_state.chat_history.append({"role": "assistant", "content": answer})

    return answer, sources


# ---------------------------
# UI
# ---------------------------
st.title("🩺 FINAL AI Diabetes System")

st.header("Enter Patient Details")

age = st.number_input("Age", 1)
gender = st.selectbox("Gender", ["Male", "Female"])
weight = st.number_input("Weight (kg)")
height = st.number_input("Height (cm)")
temp = st.number_input("Temperature")
heart = st.number_input("Heart Rate")
glucose = st.number_input("Glucose")
sys = st.number_input("Systolic BP")
dia = st.number_input("Diastolic BP")

bmi = calculate_bmi(weight, height)
st.info(f"BMI: {bmi}")

if st.button("Predict"):

    prediction = predict_diabetes(age, gender, bmi, temp, heart, glucose, sys, dia)
    risk = calculate_risk(glucose, bmi, age, heart)
    suggestion = simple_suggestion(prediction, glucose, bmi, heart)
    diet = diet_plan(glucose)

    st.success(f"Prediction: {prediction}")
    st.warning(f"Risk Score: {risk}%")
    st.info(f"Suggestion: {suggestion}")
    st.info(f"Diet: {diet}")

    record = {
        "Age": age,
        "Gender": gender,
        "BMI": bmi,
        "Glucose": glucose,
        "Prediction": prediction,
        "Risk": risk,
        "Suggestion": suggestion,
        "Diet": diet
    }

    save_patient(record)


st.subheader("Patient History")
st.dataframe(load_patients())

st.subheader("Model Accuracy")
st.write(metrics)


# ---------------------------
# CHAT
# ---------------------------
st.header("💬 Chat with AI")

question = st.text_area("Ask your question")

if st.button("Send"):
    answer, sources = rag_answer(
        question,
        {"Age": age, "BMI": bmi, "Glucose": glucose}
    )

    st.write(answer)

    with st.expander("Sources"):
        for s in sources:
            st.write(s)


# ---------------------------
# CHAT HISTORY
# ---------------------------
st.subheader("Conversation History")

for msg in st.session_state.chat_history:
    if msg["role"] == "user":
        st.write("🧑:", msg["content"])
    else:
        st.write("🤖:", msg["content"])

Writing app.py


In [10]:
import os
!pip install pyngrok
from pyngrok import ngrok
import subprocess
ngrok.set_auth_token("3AFp5iGk4vNRMrI7xiuZPOQ2lye_3pNWMx7rTHSzce6Tyj5BH")

# Terminate any existing ngrok processes to free up the session
ngrok.kill()

public_url = ngrok.connect(8501)
print("App URL:", public_url)

subprocess.Popen(["streamlit", "run", "app.py"])

App URL: NgrokTunnel: "https://c737-104-199-203-245.ngrok-free.app" -> "http://localhost:8501"


<Popen: returncode: None args: ['streamlit', 'run', 'app.py']>